In [1]:
pip install biopython pyfaidx pandas

Note: you may need to restart the kernel to use updated packages.


In [23]:
def get_sequence(chromosome, position, flank=12):
    """
    Extract a sequence of length (2*flank + 1) around the SNP position.

    Parameters:
    - chromosome: Chromosome name (e.g., 'chr1')
    - position: SNP position (1-based coordinate)
    - flank: Number of bases upstream and downstream of the SNP

    Returns:
    - sequence: DNA sequence as a string
    """
    # Adjust for 0-based indexing in pyfaidx
    start = position - flank - 1  # Subtract 1 because positions are 1-based
    end = position + flank        # End position is exclusive in slicing

    # Handle cases where start is negative
    if start < 0:
        start = 0

    # Extract the sequence
    try:
        seq = genome[chromosome][start:end]
        return str(seq).upper()
    except KeyError:
        print(f"Chromosome {chromosome} not found in the reference genome.")
        return None
    except Exception as e:
        print(f"Error retrieving sequence for {chromosome}:{position}: {e}")
        return None


# Define the padding function
def pad_sequence(seq, target_length=500):
    """
    Pads the input sequence with 'N's on both sides to reach the target length.
    The original sequence is centered. If the sequence is longer than target_length,
    it is truncated from the center.
    
    Parameters:
        seq (str): The original DNA sequence.
        target_length (int): The desired total length after padding.
    
    Returns:
        str: The padded (or truncated) sequence.
    """
    seq_length = len(seq)
    
    if seq_length >= target_length:
        # If the sequence is longer than or equal to target_length, truncate it
        # to the target_length by keeping the center part
        #start = (seq_length - target_length) // 2
        start = 0
        #end = start + target_length
        end = 500
        return seq[start:end]
    
    # Calculate total padding needed
    total_pad = target_length - seq_length
    
    # Calculate padding on each side
    pad_left = total_pad // 2
    pad_right = total_pad - pad_left
    
    # Create padded sequence
    padded_seq = 'N' * pad_left + seq + 'N' * pad_right
    return padded_seq

In [24]:
# Load SNP data
from pyfaidx import Fasta
from Bio.Seq import Seq
from Bio import SeqIO
import os
import pandas as pd
snp_data = pd.read_csv('indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv')
snp_data = snp_data[['RSID','chr','pos_hg38','Major','Minor','a1','a2']]#.drop_duplicates('RSID')
snp_data.columns = ['SNP_ID','chromosome','position','Major','Minor','a1','a2']
# Load the genome
genome = Fasta('/media/zihengc/T7/genome/hg38.fa' , as_raw=True)  # as_raw=True returns sequences as strings
# Create a new column for sequences
snp_data['Sequence'] = snp_data.apply(
    lambda row: get_sequence(row['chromosome'], row['position'], flank=250),
    axis=1
)
# Preview the updated DataFrame
missing_sequences = snp_data[snp_data['Sequence'].isnull()]
if not missing_sequences.empty:
    print("Sequences could not be retrieved for the following SNPs:")
    print(missing_sequences)
snp_data['hg38_center_base']=snp_data['Sequence'].str.slice(250,251)
snp_data['Major_len'] = snp_data['Major'].str.len()
snp_data['Minor_len'] = snp_data['Minor'].str.len()
# Create boolean mask
mask = (snp_data['Major'].str.len() == 1) & (snp_data['Minor'].str.len() == 1)
snp_data_single = snp_data[mask].copy()
# Create a boolean mask where 'Seq_13' matches 'Major' or 'Minor'
mask = snp_data_single.apply(
    lambda row: (row['hg38_center_base'] == row['Major']) or (row['hg38_center_base'] == row['Minor']),
    axis=1
)
# Filter the DataFrame
snp_data_single = snp_data_single[mask].copy()
snp_data_single['Sequence_Major'] = snp_data_single['Sequence'].str.slice(0,250)+snp_data_single['Major']+snp_data_single['Sequence'].str.slice(251,)
snp_data_single['Sequence_Minor'] = snp_data_single['Sequence'].str.slice(0,250)+snp_data_single['Minor']+snp_data_single['Sequence'].str.slice(251,)

snp_data_multi = snp_data.loc[~snp_data.index.isin(snp_data_single.index)]
snp_data_multi

,SNP_ID,chromosome,position,Major,Minor,a1,a2,Sequence,hg38_center_base,Major_len,Minor_len
143,rs1198397268,chr19,1999196,CAG,C,CAG,C,AGTGCCGGCTCTCAACAGCCTCCCCACGCCCGGAAAGCTCCGGAGC...,C,3,1
189,rs1394008105,chr2,104659015,G,GT,G,GT,CCAAGTTCGCCGGCTGTCTGGCTCTTTGTAACTCTTGATTTAGGAG...,G,1,2
190,rs1394008105,chr2,104659015,G,GT,G,GT,CCAAGTTCGCCGGCTGTCTGGCTCTTTGTAACTCTTGATTTAGGAG...,G,1,2
374,rs34158903,chr15,58727205,TC,T,TC,T,AGAAGGGAAAGTAAATGATCTAACATAACAATTAAAAGGCAAAAGC...,T,2,1
375,rs34158903,chr15,58727205,TC,T,TC,T,AGAAGGGAAAGTAAATGATCTAACATAACAATTAAAAGGCAAAAGC...,T,2,1
491,rs55706526,chr4,11039925,A,AAT,AAT,A,GCTGCCTCGGAGCTTCCTTTTTGTTGTTGTTGATGTTTGGTTTTGC...,A,1,3
572,rs67141839,chr6,32604799,G,GGACAT,GGACAT,G,CGGTTGAGACATTTTGTAGAAAAAAGAGAGAAATTTACTGGAAATC...,T,1,6
602,rs71705525,chr6,32608617,A,AACG,AACG,A,GCCAAGAATTCTCCCCTTTTCTAGTACGATCAATCATGCTTAATGA...,G,1,4
845,rs9281938,chr6,32608505,TA,T,T,TA,GTTTCGAGACCCTGTGAAAAGGTGGAGTAGGGGCAAATTATTAGCT...,T,2,1
852,rs965034941,chr19,1999195,CCA,C,CCA,C,AAGTGCCGGCTCTCAACAGCCTCCCCACGCCCGGAAAGCTCCGGAG...,C,3,1


In [30]:
snp_data['hg38_center_base']=snp_data['Sequence'].str.slice(250,251)
snp_data['Major_len'] = snp_data['Major'].str.len()
snp_data['Minor_len'] = snp_data['Minor'].str.len()
# Create boolean mask
mask = (snp_data['Major'].str.len() == 1) & (snp_data['Minor'].str.len() == 1)
snp_data_single = snp_data[mask].copy()
# Create a boolean mask where 'Seq_13' matches 'Major' or 'Minor'
mask = snp_data_single.apply(
    lambda row: (row['hg38_center_base'] == row['Major']) or (row['hg38_center_base'] == row['Minor']),
    axis=1)
# Filter the DataFrame
snp_data_single = snp_data_single[mask].copy()
snp_data_single['Sequence_Major'] = snp_data_single['Sequence'].str.slice(0,250)+snp_data_single['Major']+snp_data_single['Sequence'].str.slice(251,)
snp_data_single['Sequence_Minor'] = snp_data_single['Sequence'].str.slice(0,250)+snp_data_single['Minor']+snp_data_single['Sequence'].str.slice(251,)
snp_data_multi = snp_data.loc[~snp_data.index.isin(snp_data_single.index)]
#snp_data_multi.to_csv('MPRA_SNPCenter_Sequences_INDEL_NeedManualAnnotation500.csv')

######################mannually modified this snp_data_multi.csv########################################
snp_data_multi_modified = pd.read_csv('indexing/MPRA_SNPCenter_Sequences_INDEL_withManualAnnotation_500bp.csv',index_col=0)
snp_data_modified = pd.concat([snp_data_single,snp_data_multi_modified]).sort_index()
# Apply the padding function to 'Sequence_Major' and 'Sequence_Minor'
snp_data_modified['Sequence_Major_padded'] = snp_data_modified['Sequence_Major'].apply(pad_sequence)
snp_data_modified['Sequence_Minor_padded'] = snp_data_modified['Sequence_Minor'].apply(pad_sequence)
snp_data_modified.to_csv("indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv")

In [2]:
import pandas as pd
snp_data_modified=pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv',index_col=0)
snp_data_modified

,SNP_ID,chromosome,position,Major,Minor,a1,a2,Sequence,hg38_center_base,Major_len,Minor_len,Sequence_Major,Sequence_Minor,Padded_Sequence,Is_Valid_Length,Sequence_Major_padded,Sequence_Minor_padded
0,cg03073402,chr19,42423524,C,G,C,G,CATCCCGTTGTCCCCGCAAGCGGCAATGGGCCAGTCCGCGGACCCC...,C,1,1,CATCCCGTTGTCCCCGCAAGCGGCAATGGGCCAGTCCGCGGACCCC...,CATCCCGTTGTCCCCGCAAGCGGCAATGGGCCAGTCCGCGGACCCC...,NaN,NaN,CATCCCGTTGTCCCCGCAAGCGGCAATGGGCCAGTCCGCGGACCCC...,CATCCCGTTGTCCCCGCAAGCGGCAATGGGCCAGTCCGCGGACCCC...
1,cg03169557,chr16,89532542,C,G,C,G,CCTGAGGGGTGGCCAGGGGCCCCCGCGAGCAGCTGCCGTGTGTCTG...,C,1,1,CCTGAGGGGTGGCCAGGGGCCCCCGCGAGCAGCTGCCGTGTGTCTG...,CCTGAGGGGTGGCCAGGGGCCCCCGCGAGCAGCTGCCGTGTGTCTG...,NaN,NaN,CCTGAGGGGTGGCCAGGGGCCCCCGCGAGCAGCTGCCGTGTGTCTG...,CCTGAGGGGTGGCCAGGGGCCCCCGCGAGCAGCTGCCGTGTGTCTG...
2,cg05030077,chr16,2205198,C,G,C,G,CTGTGGTTTTGAGGGGGACCCTGGGCTCAGTGGGATGTCCTGAGGG...,C,1,1,CTGTGGTTTTGAGGGGGACCCTGGGCTCAGTGGGATGTCCTGAGGG...,CTGTGGTTTTGAGGGGGACCCTGGGCTCAGTGGGATGTCCTGAGGG...,NaN,NaN,CTGTGGTTTTGAGGGGGACCCTGGGCTCAGTGGGATGTCCTGAGGG...,CTGTGGTTTTGAGGGGGACCCTGGGCTCAGTGGGATGTCCTGAGGG...
3,cg05066959,chr8,41661790,C,G,C,G,GTAGGCCACTCCCTCTCAGCTCCACCTGCAGACAGCAGCAGAGACA...,C,1,1,GTAGGCCACTCCCTCTCAGCTCCACCTGCAGACAGCAGCAGAGACA...,GTAGGCCACTCCCTCTCAGCTCCACCTGCAGACAGCAGCAGAGACA...,NaN,NaN,GTAGGCCACTCCCTCTCAGCTCCACCTGCAGACAGCAGCAGAGACA...,GTAGGCCACTCCCTCTCAGCTCCACCTGCAGACAGCAGCAGAGACA...
4,cg05228284,chr19,2720849,C,G,C,G,GGCCAATCTACCCCAGCCCTGGGTCCCTGAAGCCCGGCTCCCGGCC...,C,1,1,GGCCAATCTACCCCAGCCCTGGGTCCCTGAAGCCCGGCTCCCGGCC...,GGCCAATCTACCCCAGCCCTGGGTCCCTGAAGCCCGGCTCCCGGCC...,NaN,NaN,GGCCAATCTACCCCAGCCCTGGGTCCCTGAAGCCCGGCTCCCGGCC...,GGCCAATCTACCCCAGCCCTGGGTCCCTGAAGCCCGGCTCCCGGCC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,rs9478143,chr6,150862035,A,G,A,G,AAAGCTTGGGTATTGTCCAAGGTTTCCCCCACTGAGACAGCCTGAG...,A,1,1,AAAGCTTGGGTATTGTCCAAGGTTTCCCCCACTGAGACAGCCTGAG...,AAAGCTTGGGTATTGTCCAAGGTTTCCCCCACTGAGACAGCCTGAG...,NaN,NaN,AAAGCTTGGGTATTGTCCAAGGTTTCCCCCACTGAGACAGCCTGAG...,AAAGCTTGGGTATTGTCCAAGGTTTCCCCCACTGAGACAGCCTGAG...
851,rs953471,chr9,124221903,G,A,G,A,CTCTCTCTTCTCCACTTACAGAGTTTTTATCCTAGAGGGGCAGCAT...,G,1,1,CTCTCTCTTCTCCACTTACAGAGTTTTTATCCTAGAGGGGCAGCAT...,CTCTCTCTTCTCCACTTACAGAGTTTTTATCCTAGAGGGGCAGCAT...,NaN,NaN,CTCTCTCTTCTCCACTTACAGAGTTTTTATCCTAGAGGGGCAGCAT...,CTCTCTCTTCTCCACTTACAGAGTTTTTATCCTAGAGGGGCAGCAT...
852,rs965034941,chr19,1999195,CCA,C,CCA,C,CTTGACTGCACCCCAGAGTCAGGTC,C,3,1,AAGTGCCGGCTCTCAACAGCCTCCCCACGCCCGGAAAGCTCCGGAG...,AAGTGCCGGCTCTCAACAGCCTCCCCACGCCCGGAAAGCTCCGGAG...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,True,AAGTGCCGGCTCTCAACAGCCTCCCCACGCCCGGAAAGCTCCGGAG...,AAGTGCCGGCTCTCAACAGCCTCCCCACGCCCGGAAAGCTCCGGAG...
853,rs983392,chr11,60156035,A,G,A,G,TTTATAAAAATAGCCTATAAAATAAAGAGATAAAAAGGAGGAAAAT...,A,1,1,TTTATAAAAATAGCCTATAAAATAAAGAGATAAAAAGGAGGAAAAT...,TTTATAAAAATAGCCTATAAAATAAAGAGATAAAAAGGAGGAAAAT...,NaN,NaN,TTTATAAAAATAGCCTATAAAATAAAGAGATAAAAAGGAGGAAAAT...,TTTATAAAAATAGCCTATAAAATAAAGAGATAAAAAGGAGGAAAAT...


In [29]:
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Define a function to create SeqRecord objects for both alleles
def create_seq_records(df, condition=None):
    """
    Creates SeqRecord objects for both Major and Minor alleles based on a condition.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing SNP data.
        condition (callable, optional): A function to filter the DataFrame rows.
        
    Returns:
        list: A list of SeqRecord objects.
    """
    seq_records = []
    seq_major = []
    seq_minor = []

    # Apply the condition if provided
    if condition:
        data_to_save = df[condition]#.drop_duplicates('RSID')
    else:
        data_to_save = df#.drop_duplicates('RSID')
    
    for index, row in data_to_save.iterrows():
        rsid = row['RSID']
        chr_ = row['chr']
        pos = row['pos_hg38']
        major_base = row['Major']
        minor_base = row['Minor']
        
        # Use only the padded sequences
        seq_major_padded = row.get('Sequence_Major_padded')
        seq_minor_padded = row.get('Sequence_Minor_padded')
        

        major_id = f"{rsid}_Major_{major_base}"
        major_description = f"{chr_}:{pos}"
        record_major = SeqRecord(
            Seq(seq_major_padded),
            id=major_id,
            description=major_description
        )
        seq_records.append(record_major)
        seq_major.append(record_major)

        minor_id = f"{rsid}_Minor_{minor_base}"
        minor_description = f"{chr_}:{pos}"
        record_minor = SeqRecord(
            Seq(seq_minor_padded),
            id=minor_id,
            description=minor_description
        )
        seq_records.append(record_minor)
        seq_minor.append(record_minor)
    return seq_records,seq_major,seq_minor


df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240813_comparative_THP1Macrophage_alleleOnly.csv', index_col=0)
snp_data_multi_modified = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA_SNPCenter_500bpSequences_withManualAnnotation_500bp.csv',index_col=0)
snp_data_multi_modified.index = df_allele.index
df_allele = pd.merge(df_allele,snp_data_multi_modified[['Sequence_Major_padded', 'Sequence_Minor_padded']],left_index=True,right_index=True)
seq_records_all = create_seq_records(df_allele)

############################################################################################
seq_records_sig,seq_major,seq_minor = create_seq_records(df_allele, condition=None)
# Write to a FASTA file
with open('/media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/Major_Minor_500bp_SNPCenter.fasta', 'w') as fasta_file:
    SeqIO.write(seq_major+seq_minor, fasta_file, 'fasta')